Title: fix_time_format.ipynb

Purpose: 

Author: Onno Nennecke on 04.2025 Modified: 23.04.2025

Input data: 

    - This file lies here: 

Output data:

    - This file lies here: 

In [1]:
import xarray as xr
import Functions.winter_date_func as winter_date_func
import glob
import os
import numpy as np
import cftime

In [2]:
# Check the time format in the netcdf files
'''
path = '/climca/people/onennecke/model_output/'
files = glob.glob(os.path.join(path, '*.nc'))
len(files)
files
for i in files:
    timeseries_ds_set = xr.open_dataset(i)
    # print(i)
    print(timeseries_ds_set['time'].dtype)
'''

"\npath = '/climca/people/onennecke/model_output/'\nfiles = glob.glob(os.path.join(path, '*.nc'))\nlen(files)\nfiles\nfor i in files:\n    timeseries_ds_set = xr.open_dataset(i)\n    # print(i)\n    print(timeseries_ds_set['time'].dtype)\n"

In [2]:
# path = '/climca/people/onennecke/model_output/bias_corrected/full_year'
path = '/climca/people/onennecke/model_output/bias_corrected_masked/full_year'
# path = '/climca/people/onennecke/model_output/not_bias_corrected/full_year'
files = sorted(glob.glob(os.path.join(path, '*.nc')))
# files = files[:60] + files[61:] # Exclude the ERA5 file
files = files[:60] + files[62:] # Exclude the ERA5 files
print(len(files))
# files

99


In [3]:
files[0][70:-3]

'ACCESS-CM2_r1i1p1f1_timeseries'

In [ ]:
ts_ds_in = xr.open_dataset(files[0])
ts_ds_win_in = winter_date_func.add_winter_calendar(ts_ds_in)
ts_win_in = ts_ds_win_in.sel(time=ts_ds_win_in['day_of_winter'].isin(range(1, 183)))


for i in files:
    print(i)
    ts_ds = xr.open_dataset(i)
    ts_ds_win = winter_date_func.add_winter_calendar(ts_ds)
    ts_win = ts_ds_win.sel(time=ts_ds_win['day_of_winter'].isin(range(1, 183)))
    old_time  = ts_win.time.values
    new_time = ts_win_in['time'].values
    ts_win = ts_win.expand_dims("ESM_run")
    # ts_win = ts_win.assign_coords(time=new_time)
    old2d = old_time[np.newaxis, :]  # shape (1, ntime)
    ts_win = ts_win.assign_coords(old_time=(("ESM_run", "time"), old2d))


    if isinstance(ts_win.time.values[0], cftime.Datetime360Day):
        print('Time format is cftime.Datetime360Day')
        cut = ts_win.isel(time = slice(2, None))
        last_day = ts_win.isel(time=-1)
        long = xr.concat([cut, last_day, last_day], dim='time')
        # Fix day_of_winter and winter_season for the two new padded days
        day_of_winter = long['day_of_winter'].values
        day_of_winter[-2] = 91
        day_of_winter[-1] = 92

        winter_year = long['winter_year'].values
        winter_season = np.array([f"{y}-{d:03d}" for y, d in zip(winter_year, day_of_winter)])

        # Reassign corrected coordinates
        ts_win = long.assign_coords(
            day_of_winter=('time', day_of_winter),
            winter_season=('time', winter_season)
        )
    if ts_win.time.dtype != 'datetime64[ns]':
        print('Time format is not datetime64[ns]')
        ts_win = ts_win.assign_coords({'time': ('time', ts_win_in.time.values)})
    print(len(ts_win.sel(time=ts_win['winter_year']==2014).time.values))
    # ts_win.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/winter_data/' + i[63:-3] + '_win.nc')
    # ts_win.to_netcdf('/climca/people/onennecke/model_output/bias_corrected/winter_data/' + i[63:-3] + '_win.nc')
    ts_win.to_netcdf('/climca/people/onennecke/model_output/bias_corrected_masked/winter_data/' + i[70:-3] + '_win.nc') ###

# print(ts_win.ESM_run.values)
# ts_win

/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ACCESS-CM2_r1i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ACCESS-CM2_r4i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ACCESS-CM2_r5i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/BCC-CSM2-MR_r1i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/CESM2_r10i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/CESM2_r11i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/CESM2_r4i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/bias_corrected_masked/full_year/EC-Earth3_r101i1p1f1_timeseries.nc
90
/climca/people/onen

In [7]:
ts_win

<xarray.Dataset> Size: 291kB
Dimensions:        (ESM_run: 1, time: 1820)
Coordinates:
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    country        float64 8B 9.0
    period         <U4 16B 'week'
    run            <U8 32B 'r9i1p1f2'
    ESM            <U11 44B 'UKESM1-0-LL'
  * ESM_run        (ESM_run) <U20 80B 'UKESM1-0-LL_r9i1p1f2'
    winter_year    (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024 2024
    day_of_winter  (time) int64 15kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
    winter_season  (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
    old_time       (ESM_run, time) object 15kB 2015-01-03 12:00:00 ... 2024-1...
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
Data variables:
    temp           (ESM_run, time) float64 15kB 5.571 4.233 5.285 ... 5.41 5.41
    demand         (ESM_run, time) float64 15kB 1.451e+03 ... 1.453e+03
    sfcWind        (ESM_run, time) float64 15kB 5.941 5.843 6.14 ... 9.436 9.436
    rsds           (ESM_run, time) float64 15kB 31.35 34.95 ... 19.66 19.66
    tas            (ESM_run, time) float64 15kB 5.522 4.568 ... 5.796 5.796
    tasmax         (ESM_run, time) float64 15kB 7.263 6.705 ... 6.983 6.983
    wind_off_prod  (ESM_run, time) float64 15kB 190.1 157.9 ... 221.2 221.2
    wind_on_prod   (ESM_run, time) float64 15kB 529.1 391.7 ... 1.374e+03
    solar_prod     (ESM_run, time) float64 15kB 88.82 106.3 ... 58.22 58.22
    total_prod     (ESM_run, time) float64 15kB 808.1 655.8 ... 1.653e+03
    Netto          (ESM_run, time) float64 15kB -642.5 -811.6 ... 200.4 200.4
    Residual_load  (ESM_run, time) float64 15kB 642.5 811.6 ... -200.4 -200.4

----

### Code for ERA5 Data

In [5]:
files = '/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ERA5_timeseries_wwd.nc'
files[70:-3]

'ERA5_timeseries_wwd'

In [9]:
# files = '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_timeseries_wwd.nc'
files = '/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ERA5_timeseries_wwd.nc'

ts_ds_ERA5 = xr.open_dataset(files)
ts_ds_win_ERA5 = winter_date_func.add_winter_calendar(ts_ds_ERA5)
ts_win_ERA5 = ts_ds_win_ERA5.sel(time=ts_ds_win_ERA5['day_of_winter'].isin(range(1, 183)))
old_time  = ts_win_ERA5.time.values
new_time = ts_win['time'].values
ts_win_ERA5 = ts_win_ERA5.expand_dims("ESM_run")
ts_win_ERA5 = ts_win_ERA5.assign_coords(time=new_time)
old2d = old_time[np.newaxis, :]  # shape (1, ntime)
ts_win_ERA5 = ts_win_ERA5.assign_coords(old_time=(("ESM_run", "time"), old2d))

# print(len(ts_win_ERA5.sel(time=ts_win_ERA5['winter_year'] == 2013).time.values))
# ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/bias_corrected/winter_data/' + files[67:-3] + '_win.nc')
# ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/winter_data/' + files[67:-3] + '_win.nc')
ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/bias_corrected_masked/winter_data/' + files[70:-3] + '_win.nc') ###
ts_win_ERA5

<xarray.Dataset> Size: 262kB
Dimensions:        (ESM_run: 1, time: 1820)
Coordinates:
    crs            int64 8B ...
    gridtype       <U6 24B ...
    country        float64 8B ...
    run            <U4 16B ...
    ESM            <U8 32B ...
  * ESM_run        (ESM_run) <U13 52B 'ERA5_hist_wwd'
    period         <U4 16B ...
    winter_year    (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024 2024
    day_of_winter  (time) int64 15kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
    winter_season  (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
    old_time       (ESM_run, time) datetime64[ns] 15kB 2015-01-01 ... 2024-12-31
Data variables:
    temp           (ESM_run, time) float64 15kB 1.323 3.335 ... 0.7396 0.4142
    demand         (ESM_run, time) float64 15kB 1.582e+03 ... 1.593e+03
    sfcWind        (ESM_run, time) float32 7kB 7.507 11.86 9.267 ... 8.049 8.775
    rsds           (ESM_run, time) float32 7kB 30.54 19.51 17.85 ... 26.3 31.92
    tas            (ESM_run, time) float32 7kB 1.771 3.587 3.046 ... 1.512 1.194
    tasmax         (ESM_run, time) float32 7kB 3.431 5.871 4.704 ... 2.709 3.361
    wind_off_prod  (ESM_run, time) float64 15kB 221.2 0.1512 ... 221.2 221.2
    wind_on_prod   (ESM_run, time) float64 15kB 1.019e+03 ... 1.187e+03
    solar_prod     (ESM_run, time) float64 15kB 91.22 46.73 45.84 ... 81.4 102.9
    total_prod     (ESM_run, time) float64 15kB 1.331e+03 ... 1.511e+03
    Netto          (ESM_run, time) float64 15kB -251.0 -34.7 ... -196.8 -81.92
    Residual_load  (ESM_run, time) float64 15kB 251.0 34.7 ... 196.8 81.92

In [10]:
# files = '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_timeseries_week.nc'
files = '/climca/people/onennecke/model_output/bias_corrected_masked/full_year/ERA5_timeseries_week.nc'


ts_ds_ERA5 = xr.open_dataset(files)
ts_ds_win_ERA5 = winter_date_func.add_winter_calendar(ts_ds_ERA5)
ts_win_ERA5 = ts_ds_win_ERA5.sel(time=ts_ds_win_ERA5['day_of_winter'].isin(range(1, 183)))
old_time  = ts_win_ERA5.time.values
new_time = ts_win['time'].values
ts_win_ERA5 = ts_win_ERA5.expand_dims("ESM_run")
ts_win_ERA5 = ts_win_ERA5.assign_coords(time=new_time)
old2d = old_time[np.newaxis, :]  # shape (1, ntime)
ts_win_ERA5 = ts_win_ERA5.assign_coords(old_time=(("ESM_run", "time"), old2d))

# print(len(ts_win_ERA5.sel(time=ts_win_ERA5['winter_year'] == 2013).time.values))
# ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/bias_corrected/winter_data/' + files[67:-3] + '_win.nc')
# ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/winter_data/' + files[67:-3] + '_win.nc')
ts_win_ERA5.to_netcdf('/climca/people/onennecke/model_output/bias_corrected_masked/winter_data/' + files[70:-3] + '_win.nc') ###
ts_win_ERA5

<xarray.Dataset> Size: 262kB
Dimensions:        (ESM_run: 1, time: 1820)
Coordinates:
    crs            int64 8B ...
    gridtype       <U6 24B ...
    country        float64 8B ...
    period         <U4 16B ...
    run            <U4 16B ...
    ESM            <U9 36B ...
  * ESM_run        (ESM_run) <U14 56B 'ERA5_hist_week'
    winter_year    (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024 2024
    day_of_winter  (time) int64 15kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
    winter_season  (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
    old_time       (ESM_run, time) datetime64[ns] 15kB 2015-01-01 ... 2024-12-31
Data variables:
    temp           (ESM_run, time) float64 15kB 1.323 3.335 ... 0.7396 0.4142
    demand         (ESM_run, time) float64 15kB 1.504e+03 ... 1.515e+03
    sfcWind        (ESM_run, time) float32 7kB 7.507 11.86 9.267 ... 8.049 8.775
    rsds           (ESM_run, time) float32 7kB 30.54 19.51 17.85 ... 26.3 31.92
    tas            (ESM_run, time) float32 7kB 1.771 3.587 3.046 ... 1.512 1.194
    tasmax         (ESM_run, time) float32 7kB 3.431 5.871 4.704 ... 2.709 3.361
    wind_off_prod  (ESM_run, time) float64 15kB 221.2 0.1512 ... 221.2 221.2
    wind_on_prod   (ESM_run, time) float64 15kB 1.019e+03 ... 1.187e+03
    solar_prod     (ESM_run, time) float64 15kB 91.22 46.73 45.84 ... 81.4 102.9
    total_prod     (ESM_run, time) float64 15kB 1.331e+03 ... 1.511e+03
    Netto          (ESM_run, time) float64 15kB -172.6 42.99 ... -118.3 -3.331
    Residual_load  (ESM_run, time) float64 15kB 172.6 -42.99 ... 118.3 3.331

In [8]:
ts_win_ERA5.load()

<xarray.Dataset> Size: 262kB
Dimensions:        (ESM_run: 1, time: 1820)
Coordinates:
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    country        float64 8B 9.0
    period         <U4 16B 'week'
    run            <U4 16B 'hist'
    ESM            <U9 36B 'ERA5_week'
  * ESM_run        (ESM_run) <U14 56B 'ERA5_hist_week'
    winter_year    (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024 2024
    day_of_winter  (time) int64 15kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
    winter_season  (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
    old_time       (ESM_run, time) datetime64[ns] 15kB 2015-01-01 ... 2024-12-31
Data variables:
    temp           (ESM_run, time) float64 15kB 1.323 3.335 ... 0.7396 0.4142
    demand         (ESM_run, time) float64 15kB 1.504e+03 ... 1.515e+03
    sfcWind        (ESM_run, time) float32 7kB 7.507 11.86 9.267 ... 8.049 8.775
    rsds           (ESM_run, time) float32 7kB 30.54 19.51 17.85 ... 26.3 31.92
    tas            (ESM_run, time) float32 7kB 1.771 3.587 3.046 ... 1.512 1.194
    tasmax         (ESM_run, time) float32 7kB 3.431 5.871 4.704 ... 2.709 3.361
    wind_off_prod  (ESM_run, time) float64 15kB 221.2 0.1512 ... 221.2 221.2
    wind_on_prod   (ESM_run, time) float64 15kB 1.019e+03 ... 1.187e+03
    solar_prod     (ESM_run, time) float64 15kB 91.22 46.73 45.84 ... 81.4 102.9
    total_prod     (ESM_run, time) float64 15kB 1.331e+03 ... 1.511e+03
    Netto          (ESM_run, time) float64 15kB -172.6 42.99 ... -118.3 -3.331
    Residual_load  (ESM_run, time) float64 15kB 172.6 -42.99 ... 118.3 3.331

-------

### Old code

In [5]:
# Old code without clipping front and appending the end
ts_ds_in = xr.open_dataset(files[0])
ts_ds_win_in = winter_date_func.add_winter_calendar(ts_ds_in)
ts_win_in = ts_ds_win_in.sel(time=ts_ds_win_in['day_of_winter'].isin(range(1, 183)))


for i in files:
    print(i)
    ts_ds = xr.open_dataset(i)
    ts_ds_win = winter_date_func.add_winter_calendar(ts_ds)
    ts_win = ts_ds_win.sel(time=ts_ds_win['day_of_winter'].isin(range(1, 183)))
    # ts_win = ts_win.assign_coords(old_time=ts_win.time)
    if ts_win.time.dtype != 'datetime64[ns]':
        ts_win = ts_win.assign_coords({'time': ('time', ts_win_in.time.values)})
    print(len(ts_win.sel(time=ts_win['winter_year']==2014).time.values))
    # ts_win.to_netcdf('/climca/people/onennecke/model_output/winter_data/' + i[38:-3] + '_win.nc')

# print(ts_win.ESM_run.values)
# ts_win

/climca/people/onennecke/model_output/ACCESS-CM2_r1i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r4i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r5i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/BCC-CSM2-MR_r1i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r10i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r11i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r4i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/EC-Earth3_r101i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r102i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r103i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r104i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3

In [11]:
import cftime
from datetime import datetime, timedelta

days = 181

# Define the starting date in the format DD.MM.
for year in range(2014, 2025):
    # Normal calendar (using datetime)
    date_normal = datetime(year, 10, 1)
    resulting_date_normal = date_normal + timedelta(days=days)
    
    # DatetimeNoLeap (365-day year)
    date_no_leap = cftime.DatetimeNoLeap(year, 10, 1)
    resulting_date_no_leap = date_no_leap + timedelta(days=days)
    
    # Datetime360Day (360-day year)
    date_360_day = cftime.Datetime360Day(year, 10, 1)
    resulting_date_360_day = date_360_day + timedelta(days=days)

    print(f"Year: {year}")
    print(f"  Normal Calendar Result:   {resulting_date_normal.strftime('%d.%m.%Y')}")
    print(f"  No Leap Year Result:      {resulting_date_no_leap.strftime('%d.%m.%Y')}")
    print(f"  360-Day Calendar Result:  {resulting_date_360_day.strftime('%d.%m.%Y')}")
    print('-' * 50)


Year: 2014
  Normal Calendar Result:   31.03.2015
  No Leap Year Result:      31.03.2015
  360-Day Calendar Result:  02.04.2015
--------------------------------------------------
Year: 2015
  Normal Calendar Result:   30.03.2016
  No Leap Year Result:      31.03.2016
  360-Day Calendar Result:  02.04.2016
--------------------------------------------------
Year: 2016
  Normal Calendar Result:   31.03.2017
  No Leap Year Result:      31.03.2017
  360-Day Calendar Result:  02.04.2017
--------------------------------------------------
Year: 2017
  Normal Calendar Result:   31.03.2018
  No Leap Year Result:      31.03.2018
  360-Day Calendar Result:  02.04.2018
--------------------------------------------------
Year: 2018
  Normal Calendar Result:   31.03.2019
  No Leap Year Result:      31.03.2019
  360-Day Calendar Result:  02.04.2019
--------------------------------------------------
Year: 2019
  Normal Calendar Result:   30.03.2020
  No Leap Year Result:      31.03.2020
  360-Day Calend